# 电商用户数据与淘宝数据清洗作业

本 Notebook 按题目要求完成两类数据的清洗。当前目录需包含淘宝文件 `淘宝全品类全国数据.xlsx`；用户数据文件可为 CSV、XLS 或 XLSX，程序会依据 `WarehouseToHome`、`OrderCount`、`CashbackAmount` 等字段自动识别。

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
SEARCH_DIRS = [BASE_DIR, Path("/mnt/data")]

TAOBAO_FILE_NAME = "淘宝全品类全国数据.xlsx"
USER_OUTPUT = BASE_DIR / "用户数据_清洗后.csv"
TAOBAO_OUTPUT = BASE_DIR / "淘宝数据_清洗后.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


## 一、通用读取与用户数据文件识别

In [ ]:
def read_table(path: Path, nrows=None):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        # 自动尝试常用中文编码
        for encoding in ("utf-8-sig", "utf-8", "gb18030"):
            try:
                return pd.read_csv(path, encoding=encoding, nrows=nrows)
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError("无法识别 CSV 编码", b"", 0, 1, str(path))
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, nrows=nrows)
    raise ValueError(f"不支持的文件格式：{path.suffix}")

def locate_file_by_name(filename: str) -> Path:
    for directory in SEARCH_DIRS:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"未找到文件：{filename}")

def locate_user_data() -> Path:
    required = {"WarehouseToHome", "OrderCount", "CashbackAmount"}
    excluded_names = {
        TAOBAO_FILE_NAME,
        USER_OUTPUT.name,
        TAOBAO_OUTPUT.name,
    }
    candidates = []
    for directory in SEARCH_DIRS:
        if not directory.exists():
            continue
        for pattern in ("*.csv", "*.xlsx", "*.xls"):
            for path in directory.glob(pattern):
                if path.name in excluded_names:
                    continue
                candidates.append(path)

    checked = set()
    for path in candidates:
        resolved = path.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        try:
            preview = read_table(path, nrows=5)
        except Exception:
            continue
        if required.issubset(set(preview.columns)):
            return path

    raise FileNotFoundError(
        "未找到用户数据文件。请将包含 WarehouseToHome、OrderCount、"
        "CashbackAmount 字段的 CSV/XLS/XLSX 文件放到 Notebook 同一目录后重新运行。"
    )


## 二、用户数据：缺失统计、中位数填补与类别统一

In [ ]:
user_df = None
user_missing_report = None
iqr_summary = None

try:
    user_path = locate_user_data()
    print(f"识别到用户数据文件：{user_path}")
    user_df = read_table(user_path)

    # 1. 每个字段的缺失数量和缺失比例
    user_missing_report = pd.DataFrame({
        "缺失数量": user_df.isna().sum(),
        "缺失比例": user_df.isna().mean()
    }).sort_values(["缺失数量", "缺失比例"], ascending=False)
    display(user_missing_report)

    # 2. 用中位数填补所有数值字段的缺失值
    numeric_cols = user_df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        median_value = user_df[col].median()
        if pd.notna(median_value):
            user_df[col] = user_df[col].fillna(median_value)

    # 3. 统一 Phone 与 Mobile Phone：统一为 Mobile Phone
    login_device_candidates = ["PreferredLoginDevice", "Preferred Login Device"]
    for col in login_device_candidates:
        if col in user_df.columns:
            user_df[col] = (
                user_df[col]
                .astype("string")
                .str.strip()
                .replace({"Phone": "Mobile Phone"})
            )

    # 4. 统一 COD 与 Cash on Delivery：统一为 Cash on Delivery
    payment_candidates = ["PreferredPaymentMode", "Preferred Payment Mode"]
    for col in payment_candidates:
        if col in user_df.columns:
            user_df[col] = (
                user_df[col]
                .astype("string")
                .str.strip()
                .replace({"COD": "Cash on Delivery"})
            )

except FileNotFoundError as exc:
    print(exc)


## 三、用户数据：IQR 候选异常值检查

In [ ]:
if user_df is not None:
    iqr_rows = []
    for col in ["WarehouseToHome", "OrderCount", "CashbackAmount"]:
        if col not in user_df.columns:
            print(f"警告：用户数据缺少字段 {col}")
            continue

        series = pd.to_numeric(user_df[col], errors="coerce")
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        flag_col = f"{col}_IQR候选异常"
        user_df[flag_col] = (series < lower) | (series > upper)

        iqr_rows.append({
            "字段": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "下界": lower,
            "上界": upper,
            "候选异常值数量": int(user_df[flag_col].sum()),
            "候选异常值比例": float(user_df[flag_col].mean()),
        })

    iqr_summary = pd.DataFrame(iqr_rows)
    display(iqr_summary)

    # 这里只标记候选异常，不直接删除，是否剔除需要业务确认
    user_df.to_csv(USER_OUTPUT, index=False, encoding="utf-8-sig")
    print(f"已导出：{USER_OUTPUT.resolve()}")


## 四、淘宝数据：商品 ID、服务字段与销量下限

In [ ]:
taobao_path = locate_file_by_name(TAOBAO_FILE_NAME)
taobao_df = read_table(taobao_path)

# 6. 清理商品 ID 中的隐藏空白字符
def clean_product_id(value):
    if pd.isna(value):
        return pd.NA
    text = str(value)
    text = re.sub(r"\\[tnr]", "", text)  # 清除字面量 \t、\n、\r
    text = re.sub(r"[\s\u200b\u200c\u200d\u2060\ufeff\xa0]+", "", text)
    return text

taobao_df["商品id"] = taobao_df["商品id"].map(clean_product_id)

# 7. “先用后付”和“退货宝”缺失值处理为“未提供”
for col in ["先用后付", "退货宝"]:
    taobao_df[col] = taobao_df[col].fillna("未提供")
    taobao_df.loc[taobao_df[col].astype(str).str.strip().eq(""), col] = "未提供"

# 8. 新建“销量下限”字段
def parse_sales_lower_bound(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().replace(",", "")
    match = re.search(r"(\d+(?:\.\d+)?)\s*(万)?\s*\+", text)
    if match is None:
        match = re.search(r"(\d+(?:\.\d+)?)\s*(万)?", text)
    if match is None:
        return pd.NA
    number = float(match.group(1))
    if match.group(2):
        number *= 10000
    return int(number) if number.is_integer() else number

taobao_df["销量下限"] = taobao_df["商品销量"].map(parse_sales_lower_bound).astype("Int64")

display(taobao_df.head())
print(taobao_df[["先用后付", "退货宝", "销量下限"]].isna().sum())


## 五、导出两个清洗后的 CSV

In [ ]:
taobao_df.to_csv(TAOBAO_OUTPUT, index=False, encoding="utf-8-sig")
print(f"已导出：{TAOBAO_OUTPUT.resolve()}")

if user_df is None:
    print(
        "用户数据源文件尚未提供，因此本次不会伪造“用户数据_清洗后.csv”；"
        "补充用户数据文件并重新运行后，程序会自动生成第二个 CSV。"
    )
else:
    print(f"两个 CSV 均已生成：{USER_OUTPUT.name}、{TAOBAO_OUTPUT.name}")


我完成了缺失统计与数值中位数填补、类别名称统一、IQR 候选异常标记，并清理了淘宝商品 ID、服务字段缺失值和销量下限。中位数用于降低极端值对填补结果的影响，类别统一便于统计，IQR 只做候选标记以避免未经判断直接删除有效数据。WarehouseToHome、OrderCount、CashbackAmount 的候选异常是否真实异常，以及“未提供”是否应与“未开通”区分，仍需要业务人员确认。